In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType, DateType


In [0]:
# Configurazione variabili
source_path = "abfss://source@storageaccountmeteo.dfs.core.windows.net/"
checkpoint_path = "abfss://checkpoint@storageaccountmeteo.dfs.core.windows.net/"


## acheck adls folders

In [0]:
dbutils.fs.ls("abfss://source@storageaccountmeteo.dfs.core.windows.net/")

[FileInfo(path='abfss://source@storageaccountmeteo.dfs.core.windows.net/checkpoint/', name='checkpoint/', size=0, modificationTime=1774951696000),
 FileInfo(path='abfss://source@storageaccountmeteo.dfs.core.windows.net/tables/', name='tables/', size=0, modificationTime=1774951705000)]

In [0]:
dbutils.fs.ls("abfss://source@storageaccountmeteo.dfs.core.windows.net/")

[FileInfo(path='abfss://source@storageaccountmeteo.dfs.core.windows.net/checkpoint/', name='checkpoint/', size=0, modificationTime=1774951696000),
 FileInfo(path='abfss://source@storageaccountmeteo.dfs.core.windows.net/tables/', name='tables/', size=0, modificationTime=1774951705000)]

In [0]:
dbutils.fs.ls("abfss://source@storageaccountmeteo.dfs.core.windows.net/tables/")


[FileInfo(path='abfss://source@storageaccountmeteo.dfs.core.windows.net/tables/weather_milan_2024.csv', name='weather_milan_2024.csv', size=4321, modificationTime=1774951740000),
 FileInfo(path='abfss://source@storageaccountmeteo.dfs.core.windows.net/tables/weather_rome_2024.csv', name='weather_rome_2024.csv', size=4259, modificationTime=1774951740000),
 FileInfo(path='abfss://source@storageaccountmeteo.dfs.core.windows.net/tables/weather_stations_metadata.csv', name='weather_stations_metadata.csv', size=2334, modificationTime=1774951740000)]

In [0]:
for t in tabelle:
    dbutils.fs.mv(
        f"abfss://source@storageaccountmeteo.dfs.core.windows.net/tables/{t}.csv",
        f"abfss://source@storageaccountmeteo.dfs.core.windows.net/tables/{t}/{t}.csv"
    )
    print(f"✅ Moved: {t}")

✅ Moved: weather_milan_2024
✅ Moved: weather_rome_2024
✅ Moved: weather_stations_metadata


In [0]:
dbutils.fs.ls("abfss://source@storageaccountmeteo.dfs.core.windows.net/tables/")

[FileInfo(path='abfss://source@storageaccountmeteo.dfs.core.windows.net/tables/weather_milan_2024/', name='weather_milan_2024/', size=0, modificationTime=1774952566000),
 FileInfo(path='abfss://source@storageaccountmeteo.dfs.core.windows.net/tables/weather_rome_2024/', name='weather_rome_2024/', size=0, modificationTime=1774952567000),
 FileInfo(path='abfss://source@storageaccountmeteo.dfs.core.windows.net/tables/weather_stations_metadata/', name='weather_stations_metadata/', size=0, modificationTime=1774952567000)]

autoloader

In [0]:
tabelle = ["weather_milan_2024", "weather_rome_2024", "weather_stations_metadata"]
source_base_path     = "abfss://source@storageaccountmeteo.dfs.core.windows.net/tables"
checkpoint_base_path = "abfss://source@storageaccountmeteo.dfs.core.windows.net/checkpoint"

queries = []

for t in tabelle:
    print(f"Avvio ingestion per: {t}")

    s_path      = f"{source_base_path}/{t}/"
    schema_path = f"{checkpoint_base_path}/schema/{t}/"
    c_path      = f"{checkpoint_base_path}/checkpoint/{t}/"

    q = (spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("cloudFiles.schemaLocation", schema_path)
        .option("header", "true")
        .option("cloudFiles.inferColumnTypes", "true")
        .load(s_path)
        .writeStream
        .option("checkpointLocation", c_path)
        .trigger(availableNow=True)
        .toTable(f"catalogmeteo.bronze.{t}")
    )
    queries.append(q)

for q in queries:
    q.awaitTermination()

Avvio ingestion per: weather_milan_2024
Avvio ingestion per: weather_rome_2024
Avvio ingestion per: weather_stations_metadata


check format

In [0]:
spark.sql("DESCRIBE DETAIL catalogmeteo.bronze.weather_milan_2024").select("format").show()

+------+
|format|
+------+
| delta|
+------+

